# Case 8

# Does window stride matter more for MLP-VAE-Cyclic than for Transformer-VAE?

## Purpose of this notebook

This notebook demonstrates one of the window-size-and-stride findings of the thesis:

> Stride controls how far a window's starting hour advances between consecutive windows, and therefore how consistently each window lands at the same phase of the diurnal cycle. Architectures whose features assume a stable phase can be unusually sensitive to this, even when the window size itself is unchanged.

The thesis studies anomaly detection in streaming ERA5 weather data. It compares the impact of different configurations (VAE architecture, streaming and feature manipulation methods, window size and stride) on VAEs with different deep learning models as encoder/decoder.

This testcase focuses on **contextual anomalies** with a fixed 1-week window (`seq_len=168`) and compares stride sensitivity across two architectures: **MLP-VAE-Cyclic** (strides 12, 24, 48) and **Transformer-VAE** (strides 6, 12, 24).

---

For the complete experiment definitions, consult the thesis section associated with **4.4.5 Effects of Window Size and Stride** (Table 4.20, Figures 4.20 and 4.21), together with the YAML configuration files used by this suite.

## What is being compared?

The notebook runs six variants — two architectures, each swept over its own set of strides:

| Variant | Stride | Meaning |
|---|---:|---|
| **MLP-VAE-Cyclic** | 12 | Windows advance half a day at a time — more overlap, more phase drift than the default. |
| **MLP-VAE-Cyclic** | 24 | Default: windows advance exactly one day at a time, staying phase-aligned to the diurnal cycle. |
| **MLP-VAE-Cyclic** | 48 | Windows advance two days at a time — still phase-aligned (24 divides 48), but half as many windows seen. |
| **Transformer-VAE** | 6 | Windows advance a quarter-day at a time — the most overlap and phase drift tested for this architecture. |
| **Transformer-VAE** | 12 | Windows advance half a day at a time. |
| **Transformer-VAE** | 24 | Default: windows advance exactly one day at a time. |

The two architectures are swept over different stride ranges because the effect being tested differs: MLP-VAE-Cyclic's explicit cyclic (hour-of-day, day-of-year) features assume every window starts at a consistent phase, so the comparison is between phase-aligned and phase-drifting strides. Transformer-VAE has no such explicit phase assumption, so the comparison instead spans a range including finer strides to see whether stride matters at all in the absence of that assumption.

## Findings being illustrated

The thesis found that MLP-VAE-Cyclic is unusually sensitive to stride because its cyclic features (sine/cosine encodings of hour-of-day and day-of-year) implicitly assume a stable relationship between window position and calendar phase. When stride keeps windows phase-aligned (e.g. stride=24, advancing exactly one day), that assumption holds; when it doesn't, the cyclic features can become a source of noise rather than useful context. Transformer-VAE, without an equivalent explicit phase assumption, is expected to be comparatively insensitive to the same stride changes.

This reduced notebook uses the full-scale ERA5 export by default (see `case01_clean_baseline.ipynb` for why), so results should be directionally close to the thesis, though exact metric values are not expected to match (single seed, this repo's own port of the training code). The important result is the **relative pattern**: MLP-VAE-Cyclic's F1 should vary more across its stride sweep than Transformer-VAE's does across its own.

---

## What this notebook will do

1. Check whether a GPU is available.
2. Clone and install the thesis repository.
3. Run the three MLP-VAE-Cyclic stride variants and the three Transformer-VAE stride variants on the same contextual-anomaly dataset.
4. Collect F1, AUC, precision, recall, and confusion-matrix counts.
5. Plot F1 across each architecture's own stride sweep.

### Expected runtime

Window-based runs use the GPU when available. All six variants share the same dataset and window size; only stride differs within each architecture.

### Before running

This notebook reads code and data from a **private GitHub repository**. You must:

1. Have permission to access the read-only repository with a fine-grained GitHub token.
2. In Colab, open the **Secrets** panel using the key icon.
3. Add the secret `GITHUB_TOKEN`.
4. Enable **Notebook access** for that secret.
5. Select **Runtime → Change runtime type → GPU**.
6. Choose **Runtime → Run all**.

The source ERA5 data and its licence information are described in `DATA_LICENSE.md`.

# Run Case

### Check GPU availability

In [1]:
%matplotlib inline
import torch

print("Environment check")
print("-----------------")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Selected device:", torch.cuda.get_device_name(0))
else:
    print(
        "No GPU was detected. The notebook can still run, but "
        "window-based training may be slower."
    )


Environment check
-----------------
PyTorch version: 2.10.0+cu126
CUDA available: True
Selected device: Quadro RTX 6000


### Import and/or load Repo

In [2]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/hadasecohen/streaming-vae-anomaly-detection.git"
REPO_DIR = Path("/content/repo") if "COLAB_RELEASE_TAG" in os.environ else Path("repo")

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise RuntimeError(
            "The Colab secret GITHUB_TOKEN is unavailable. "
            "Open the key icon in the left sidebar, add the token, "
            "and enable Notebook access."
        )

    if not REPO_DIR.exists():
        # Use an askpass helper so the token is not stored in the Git remote URL.
        askpass = Path("/content/git_askpass.sh")
        askpass.write_text(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) echo "x-access-token" ;;\n'
            '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
            'esac\n'
        )
        askpass.chmod(0o700)

        env = os.environ.copy()
        env["GITHUB_TOKEN"] = token
        env["GIT_ASKPASS"] = str(askpass)
        env["GIT_TERMINAL_PROMPT"] = "0"

        try:
            subprocess.run(
                ["git", "clone", REPO_URL, str(REPO_DIR)],
                check=True,
                env=env,
            )
        finally:
            askpass.unlink(missing_ok=True)

    os.chdir(REPO_DIR)
else:
    # When launched from notebooks/cases inside a local checkout,
    # move to the repository root.
    if not Path("run_regression.py").exists():
        os.chdir("../..")

print("Working directory:", os.getcwd())
print("Installing the project dependencies...")
subprocess.run(
    ["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
%env MPLBACKEND=Agg
print("Setup complete.")


Working directory: /home/cohenhada/streaming-vae-anomaly-detection
Installing the project dependencies...
env: MPLBACKEND=Agg
Setup complete.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


### Step 1 — Configure the run

In [3]:
import os
import yaml

CASE_ID = "case08_stride_alignment"
SESSION_DIR = "runs/regression"
OUTPUT_DIR = f"{SESSION_DIR}/{CASE_ID}"

# Edit this directly — works the same locally and on Colab:
#   "full_split_files" (default) -> anom_types/contextual.yaml's own default —
#                      full-scale contextual data, pre-split into a
#                      warmup-only file (shared with case01 — the warmup
#                      portion is anomaly-free and identical regardless of
#                      anomaly type) and a test-only file with the
#                      contextual anomalies injected. Committed to the repo.
#   "mini_50pct"    -> the in-repo 377,784-row ERA5 slice (~50% of full-scale).
#                      Committed to the repo.
#   "mini_28pct"    -> the in-repo 210,384-row ERA5 slice (~28% of full-scale).
#                      Committed to the repo.
DATA_SOURCE = "full_split_files"

_DATA_SOURCE_LAYERS = {
    "full_split_files": None,   # anom_types/contextual.yaml's own default — no extra layer needed
    "mini_50pct": "../../modules/data_source/mini_50pct_contextual.yaml",
    "mini_28pct": "../../modules/data_source/mini_28pct_contextual.yaml",
}

_suite_source_path = f"notebooks/cases/{CASE_ID}_suite.yaml"
_suite_dir = os.path.dirname(os.path.abspath(_suite_source_path))

def _resolve(path):
    # base_config / config_layers entries are relative paths meant to be
    # read relative to the suite file's own directory (notebooks/cases/).
    # Resolving them to absolute paths here — rather than leaving them
    # relative — means the resolved copy stays correct no matter how deep
    # under OUTPUT_DIR it ends up being written.
    return path if os.path.isabs(path) else os.path.normpath(os.path.join(_suite_dir, path))

_data_source_layer = _DATA_SOURCE_LAYERS[DATA_SOURCE]

with open(_suite_source_path) as f:
    _suite = yaml.safe_load(f)

_suite["base_config"] = _resolve(_suite["base_config"])
for _run in _suite["runs"]:
    _layers = [_resolve(p) for p in _run["config_layers"]]
    if _data_source_layer:
        _layers.append(_resolve(_data_source_layer))
    _run["config_layers"] = _layers

# Written under OUTPUT_DIR (runs/, already gitignored and read-write) rather
# than notebooks/ (source-controlled, meant to stay read-only) — os.makedirs
# because OUTPUT_DIR won't exist yet on a fresh run.
os.makedirs(OUTPUT_DIR, exist_ok=True)
SUITE_PATH = f"{OUTPUT_DIR}/{CASE_ID}_suite_resolved.yaml"
with open(SUITE_PATH, "w") as f:
    yaml.safe_dump(_suite, f, sort_keys=False)

print(f"DATA_SOURCE = {DATA_SOURCE!r} -> {_data_source_layer}")

DATA_SOURCE = 'full_split_files' -> None


#### Configuration file

The notebooks test suites combines a shared ERA5 configuration (`modules/era5_common.yaml`) with the architecture, anomaly type, stream mode, and stride settings for each run.

`DATA_SOURCE` may be configured to choose other datasets from the era5 data directory.
The default is `"full_split_files"`, which is the full-scale contextual-anomaly data, where warmup and test files are split (`data/era5/full_scale/split/era5_clean_warmup.csv` and `data/era5/full_scale/split/era5_contextual_test.csv`).

All variants use the same base configuration, window size, and evaluation procedure. The intended comparison is therefore the effect of **stride** on each architecture's own contextual-anomaly detection.

#### Output Directory

During execution, the script creates a separate run directory in the repo for each variant under `OUTPUT_DIR` (`runs/regression/case08_stride_alignment` by default).

`notebooks/cases/case08_stride_alignment_suite.yaml` layers `modules/era5_common.yaml` with the architecture / anomaly-type / stream-mode / feature-engineering fragments and the per-run stride override (see the suite file for the exact overrides).

### Step 2 Run the testsuite

### Command:

In [ ]:
!python run_regression.py {SUITE_PATH} \
    --session {SESSION_DIR} --skip-existing

[suite] log     : /home/cohenhada/streaming-vae-anomaly-detection/runs/regression/suite.log
[suite] suite   : /home/cohenhada/streaming-vae-anomaly-detection/runs/regression/case08_stride_alignment/case08_stride_alignment_suite_resolved.yaml
[suite] session : /home/cohenhada/streaming-vae-anomaly-detection/runs/regression
[suite] base cfg: /home/cohenhada/streaming-vae-anomaly-detection/modules/era5_common.yaml
[suite] module  : regression.run_trial
[suite] GPUs    : [0, 1, 2]  (3 slot(s))
[suite] 6 run(s) planned:
  [  1] MLP_Cyclic_stride_12  [case08_stride_alignment/case08_stride_alignment_suite_resolved]  (module: regression.run_trial)
  [  2] MLP_Cyclic_stride_24  [case08_stride_alignment/case08_stride_alignment_suite_resolved]  (module: regression.run_trial)
  [  3] MLP_Cyclic_stride_48  [case08_stride_alignment/case08_stride_alignment_suite_resolved]  (module: regression.run_trial)
  [  4] TF_VAE_stride_6  [case08_stride_alignment/case08_stride_alignment_suite_resolved]  (module

### Step 3 — Build a common comparison

#### The next command reads the predictions from every run and produces:

- a common performance table
- anomaly-score histograms
- an F1 comparison across variants
- confusion-matrix summaries
- seed-stability diagnostics, where applicable.

These outputs are saved under `OUTPUT_DIR/cross_compare/`.

### Command:

In [ ]:
!python cross_compare.py {OUTPUT_DIR}

### Step 4 — Quantitative Results

### Command:

In [ ]:
import pandas as pd
from IPython.display import display

performance_path = f"{OUTPUT_DIR}/cross_compare/performance_table.csv"
perf = pd.read_csv(performance_path)

columns = [
    "run_name", "arch", "anomaly", "variant",
    "f1", "auc", "precision", "recall",
    "tp", "fp", "fn",
]

perf["variant"] = perf["variant"].astype(int)

print("Comparison of stride variants")
display(
    perf[columns]
    .sort_values(["arch", "variant"])
    .reset_index(drop=True)
    .style.format({
        "f1": "{:.3f}",
        "auc": "{:.3f}",
        "precision": "{:.3f}",
        "recall": "{:.3f}",
    })
)

#### Focus first on `f1`, `precision`, and `recall`. These are the main thesis metrics:

- higher **precision** means fewer normal observations were falsely flagged;
- higher **recall** means more contextual anomalies were detected;
- `tp`, `fp`, and `fn` show the corresponding counts.

#### Expected pattern

```text
MLP-VAE-Cyclic:  F1 varies noticeably across strides 12 / 24 / 48
                 (phase-aligned strides expected to outperform phase-drifting ones)
Transformer-VAE: F1 comparatively flat across strides 6 / 12 / 24
```

The testcase is successful when MLP-VAE-Cyclic's F1 spread across its stride sweep is clearly larger than Transformer-VAE's spread across its own, even if the exact values differ from the thesis.

### Step 5 — Qualitative support (Plots)

#### F1 by stride, per architecture

##### Command:

In [ ]:
!python scripts/plot_categorical_bars.py {OUTPUT_DIR} --section mlp_cyclic_stride --metric f1
!python scripts/plot_categorical_bars.py {OUTPUT_DIR} --section tf_vae_stride --metric f1

import glob
from IPython.display import Image, display

for p in sorted(glob.glob(f"{OUTPUT_DIR}/cross_compare/bars_*_f1_*.png")):
    display(Image(filename=p))

Two bar charts — one for MLP-VAE-Cyclic across strides 12/24/48, one for Transformer-VAE across strides 6/12/24. `scripts/plot_categorical_bars.py` is the same script used in `case06_reconstruction_loss.ipynb` and `case07_point_vs_window.ipynb`.

#### The plots can indicate, among other things:

1. Whether MLP-VAE-Cyclic's F1 is visibly higher at phase-aligned strides (24, 48) than at stride 12.
2. Whether Transformer-VAE's F1 stays comparatively flat across its own stride sweep.
3. Whether the overall spread (max − min F1) is larger for MLP-VAE-Cyclic than for Transformer-VAE.

### Main takeaway

Stride is not just a speed/coverage knob — for an architecture whose features implicitly assume a stable relationship between window position and calendar phase, it can also determine whether that assumption holds. MLP-VAE-Cyclic's cyclic time features are computed relative to each window, so a stride that keeps windows phase-aligned preserves the intended meaning of those features, while a phase-drifting stride can undermine it. Transformer-VAE, without an equivalent explicit phase assumption, is expected to be comparatively insensitive to the same stride changes.

## Scope of this testcase

This notebook is a compact demonstration, not a full reproduction of every window-size-and-stride experiment reported in the thesis. With `DATA_SOURCE = "full_split_files"` (the default) it runs on the full-scale contextual-anomaly data. With `DATA_SOURCE = "mini_50pct"` / `"mini_28pct"` a shorter ERA5 interval is used instead — useful for a quick check, but see `case01_clean_baseline.ipynb` for why it can give different results than the full-scale data.

This notebook only sweeps stride, at a single fixed window size (`seq_len=168`, 1 week). The thesis's own treatment (section 4.4.5, Table 4.20, Figures 4.20 and 4.21) also varies window size itself, which this notebook does not.

Conclusions should be based on the **relative spread of F1 across each architecture's own stride sweep**, rather than exact numerical agreement with the thesis figures (single seed, this repo's own port of the training code).